# Chapter 6: Case Study 1

##  Step 1: Load packages

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import sys 
import regex as re 
from glob import glob
import tomli
import os

## Step 2: Merge Snipseq outputs

In [3]:
#results from rep7-rep10
df10 = pd.read_csv('results/snipseq/bulk/SRR30861029_2_ss_assign_ret.csv')
df9 = pd.read_csv('results/snipseq/bulk/SRR30861030_2_ss_assign_ret.csv')
df8 = pd.read_csv('results/snipseq/bulk/SRR30861031_2_ss_assign_ret.csv')
df7 = pd.read_csv('results/snipseq/bulk/SRR30861032_2_ss_assign_ret.csv')
df6 = pd.read_csv('results/snipseq/bulk/SRR30861033_2_ss_assign_ret.csv')
df5 = pd.read_csv('results/snipseq/bulk/SRR30861034_2_ss_assign_ret.csv')
df4 = pd.read_csv('results/snipseq/bulk/SRR30861035_2_ss_assign_ret.csv')
df3 = pd.read_csv('results/snipseq/bulk/SRR30861036_2_ss_assign_ret.csv')

ss_ret = pd.concat([df10, df9, df8, df7, df6, df5, df4, df3])

srr_map = {
    'SRR30861029': '10',
    'SRR30861030': '9',
    'SRR30861031': '8',
    'SRR30861032': '7',
    'SRR30861033': '6',
    'SRR30861034': '5',
    'SRR30861035': '4',
    'SRR30861036': '3',
    'SRR30861037': '2',
    'SRR30861038': '1'}

ss_ret['srr'] = ss_ret['read_id'].str.split('.', n=1, expand=True)[0]

mapped = ss_ret['srr'].map(srr_map)
ss_ret['replicate'] = pd.DataFrame(mapped.tolist(), index=ss_ret.index)
ss_ret = ss_ret.rename(columns={'feature':'Barcode'})
ss_ret['pair'] = ss_ret['pair'].replace({'dtom': 'dTom','gfp_s': 'GFP','gfp_uns': 'unspliced'})
ss_ret['sample'] = ss_ret.pair + '_' + ss_ret.replicate
ss_ret['source'] = 'Snipseq'

ss_ret_matrix = (ss_ret.pivot_table(index='Barcode',columns='sample',values='read_id',aggfunc='size',fill_value=0).rename_axis(index='Barcode').reset_index())
# ss_ret_matrix.to_csv('results/snipseq/bulk/snipseq_matrix_r3_r10.csv', index=False)

In [10]:
# rep1 and 2 file are too big so I created the matrix separately
ss_ret_matrix_r1 = pd.read_csv('results/snipseq/bulk/snipseq_matrix_r1.csv')
ss_ret_matrix_r2 = pd.read_csv('results/snipseq/bulk/snipseq_matrix_r2.csv')
ss_ret_matrix_r3_r10 = pd.read_csv('results/snipseq/bulk/snipseq_matrix_r3_r10.csv')
# merge all matrix together and save
ss_ret_matrix_r1_r2 = pd.merge(ss_ret_matrix_r1, ss_ret_matrix_r2, on ='Barcode')
ss_ret_matrix = pd.merge(ss_ret_matrix_r1_r2, ss_ret_matrix_r3_r10, on ='Barcode')
# ss_ret_matrix.to_csv('results/snipseq/bulk/snipseq_matrix_final.csv', index=False)

,Barcode,GFP_1,dTom_1,unspliced_1,GFP_2,dTom_2,unspliced_2,GFP_10,GFP_3,GFP_4,...,dTom_4,dTom_5,dTom_6,dTom_7,dTom_8,dTom_9,unspliced_10,unspliced_3,unspliced_4,unspliced_5
0,AAAAAAAAAAAAAAAAAA,18,0,0,2,0,0,0,1,2,...,0,0,0,0,0,0,0,0,0,0
1,AAAAAAAAAAAAAAAAAC,8,1,0,3,1,0,0,0,3,...,1,0,0,0,0,0,0,0,0,0
2,AAAAAAAAAAAAAAAAAT,6,0,0,3,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
3,AAAAAAAAAAAAAAAACA,2,0,0,1,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,AAAAAAAAAAAAAAAACC,19,7,0,5,4,0,0,0,3,...,3,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
366239,TTTTTTTTCTTCCTTACT,1,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
366240,TTTTTTTTTCTAACTTTA,273,0,0,109,0,0,0,0,2,...,0,0,0,0,0,0,0,0,0,0
366241,TTTTTTTTTCTCACTTTA,2,90,0,0,10,0,0,0,0,...,3,0,0,0,0,0,0,0,0,0
366242,TTTTTTTTTTCTAACTTT,490,0,0,107,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [13]:
ss_ret_matrix.columns

Index(['Barcode', 'GFP_1', 'dTom_1', 'unspliced_1', 'GFP_2', 'dTom_2',
       'unspliced_2', 'GFP_10', 'GFP_3', 'GFP_4', 'GFP_5', 'GFP_6', 'GFP_7',
       'GFP_8', 'GFP_9', 'dTom_10', 'dTom_3', 'dTom_4', 'dTom_5', 'dTom_6',
       'dTom_7', 'dTom_8', 'dTom_9', 'unspliced_10', 'unspliced_3',
       'unspliced_4', 'unspliced_5'],
      dtype='str')

## Step 3: Load results

### Load merged Snipseq output

In [2]:
ss_ret_matrix = pd.read_csv('results/snipseq/bulk/snipseq_matrix_final.csv')

In [3]:
ss_long = pd.wide_to_long(ss_ret_matrix,
                          stubnames=["GFP", "dTom", "unspliced"],
                          i="Barcode",
                          j="sample",
                          sep="_",
                          suffix=r"\d+").reset_index()

# Filter results
mask = (ss_long["GFP"] >= 10) & (ss_long["dTom"] >= 10)
passed = ss_long[mask].groupby("Barcode")["sample"].nunique()
keep_barcodes = passed[passed >= 3].index

filtered_ss = ss_ret_matrix.loc[ss_ret_matrix.Barcode.isin(keep_barcodes)].copy()

filtered_ss = filtered_ss[['Barcode', 'GFP_1', 'GFP_2', 'GFP_3', 'GFP_4', 'GFP_5', 'GFP_6',
                           'GFP_7', 'GFP_8', 'GFP_9', 'GFP_10', 'dTom_1', 'dTom_2', 'dTom_3',
                           'dTom_4', 'dTom_5', 'dTom_6', 'dTom_7', 'dTom_8', 'dTom_9', 'dTom_10',
                           'unspliced_1', 'unspliced_2', 'unspliced_3', 'unspliced_4',
                           'unspliced_5', 'unspliced_10']]

# Keep Barcode as id; melt all other columns
id_cols = ['Barcode']
value_cols = [c for c in filtered_ss.columns if c not in id_cols]

ss_long = filtered_ss.melt(
    id_vars=id_cols,
    value_vars=value_cols,
    var_name='measurement',
    value_name='counts'
)

# Split 'GFP_1' into ['GFP', 'rep1'] using simple string ops
parts = ss_long['measurement'].str.split('_', n=1, expand=True)
parts.columns = ['pair', 'rep_token']

# Convert rep_token like 'rep1' -> 1 without regex
parts['replicate'] = parts['rep_token'].str.replace('rep', '', regex=False).astype('int64')

# Assemble tidy DataFrame
ss_long = pd.concat([ss_long.drop(columns=['measurement']), parts[['replicate', 'pair']]],axis=1)

ss_long = ss_long[['Barcode', 'replicate', 'pair', 'counts']]
ss_long['source'] = 'Snipseq'

In [12]:
ss_ret_matrix.shape

(366244, 27)

### Load results from IME study

In [4]:
kw = pd.read_csv('metadata/kowal_s5.csv')

kw_long = pd.wide_to_long(kw,
                          stubnames=["GFP", "dTom", "unspliced"],
                          i="Barcode",
                          j="sample",
                          sep="_",
                          suffix=r"\d+").reset_index()

mask = (kw_long["GFP"] >= 10) & (kw_long["dTom"] >= 10)
passed = kw_long[mask].groupby("Barcode")["sample"].nunique()
keep_barcodes = passed[passed >= 3].index

filtered_kw = kw.loc[kw.Barcode.isin(keep_barcodes)].copy()

filtered_kw = filtered_kw[['Barcode', 'GFP_1', 'GFP_2', 'GFP_3', 'GFP_4', 'GFP_5', 'GFP_6',
                           'GFP_7', 'GFP_8', 'GFP_9', 'GFP_10', 'dTom_1', 'dTom_2', 'dTom_3',
                           'dTom_4', 'dTom_5', 'dTom_6', 'dTom_7', 'dTom_8', 'dTom_9', 'dTom_10',
                           'unspliced_1', 'unspliced_2', 'unspliced_3', 'unspliced_4',
                           'unspliced_5', 'unspliced_6', 'unspliced_7', 'unspliced_8',
                           'unspliced_9', 'unspliced_10']]

# Keep Barcode as id; melt all other columns
id_cols = ['Barcode']
value_cols = [c for c in filtered_kw.columns if c not in id_cols]

kw_long = filtered_kw.melt(
    id_vars=id_cols,
    value_vars=value_cols,
    var_name='measurement',
    value_name='counts'
)

# Split 'GFP_1' into ['GFP', 'rep1'] using simple string ops
parts = kw_long['measurement'].str.split('_', n=1, expand=True)
parts.columns = ['pair', 'rep_token']

# Convert rep_token like 'rep1' -> 1 without regex
parts['replicate'] = parts['rep_token'].str.replace('rep', '', regex=False).astype('int64')

# Assemble tidy DataFrame
kw_long = pd.concat([kw_long.drop(columns=['measurement']), parts[['replicate', 'pair']]],axis=1)

kw_long = kw_long[['Barcode', 'replicate', 'pair', 'counts']]
kw_long['source'] = 'Kowal'

In [11]:
kw.shape

(35599, 31)

## Step 4: Visualising results

In [10]:
df_long = pd.concat([ss_long, kw_long])
df_long['pair'] = df_long['pair'].replace({'unspliced':'GFP_uns'})
df_long['source'] = df_long['source'].replace({'kw':'Kowal','ss':'Snipseq'})
df_long = df_long[~df_long.Barcode.str.contains('G')] # remove barcodes with G
df_long.shape

(1858704, 5)

In [11]:
df_long_copy = df_long.copy()
df_long_copy = df_long_copy[df_long_copy.counts > 0]

barcode_sources = df_long_copy.groupby(
    ['replicate', 'pair', 'Barcode']
)['source'].unique()

def classify(sources):
    sources = set(sources)
    if sources == {'Kowal', 'Snipseq'}:
        return 'both'
    elif sources == {'Kowal'}:
        return 'kw_only'
    elif sources == {'Snipseq'}:
        return 'ss_only'
    else:
        return 'other'

barcode_class = barcode_sources.apply(classify).reset_index()
barcode_class = barcode_class.rename(columns={0: 'barcode_group'})

df_long_copy = df_long_copy.merge(
    barcode_class,
    on=['replicate','pair', 'Barcode'],
    how='left'
)

### Figure 6.4(A)

In [14]:
number_of_unique_barcode = df_long_copy.groupby(['source_y'])['Barcode'].nunique().reset_index(name='n_barcodes')
print(number_of_unique_barcode)

plt.figure(figsize=(5,4))
sns.barplot(number_of_unique_barcode, x='source_y',y='n_barcodes')
# plt.savefig('results/snipseq/bulk/figures/barcode_detection_overall_dodgedbars.png', dpi=300)
# plt.savefig('results/snipseq/bulk/figures/barcode_detection_overall_dodgedbars.pdf', dpi=300)

### Figure 6.4(B)

In [13]:
number_of_unique_barcode_per_rep = df_long_copy.groupby(['replicate', 'source_y'])['Barcode'].nunique().reset_index(name='n_barcodes')
print(number_of_unique_barcode_per_rep)

plt.figure(figsize=(10,4))
sns.barplot(number_of_unique_barcode_per_rep, x='replicate',y='n_barcodes',hue='source_y')
# plt.savefig('results/snipseq/bulk/figures/barcode_detection_per_rep_dodgedbars.png', dpi=300)
# plt.savefig('results/snipseq/bulk/figures/barcode_detection_per_rep_dodgedbars.pdf', dpi=300)

### Figure 6.4(C)

In [17]:
#number of reads recovered
df_plot_reads = (df_long_copy[df_long_copy.source_y=='both'].groupby(['pair','source_x'])['counts'].sum().reset_index())

plt.figure(figsize=(5,4))

ax = sns.barplot(
    data=df_plot_reads,
    x='pair',
    y='counts',
    order=['dTom', 'GFP'],
    hue='source_x'
)

# annotate bars
for container in ax.containers:
    ax.bar_label(container, fmt='%d', fontsize=8)

plt.tight_layout()


# plt.savefig('results/snipseq/bulk/figures/reads_detection_per_reporter_dodgedbars.png', dpi=300)
# plt.savefig('results/snipseq/bulk/figures/reads_detection_per_reporter_dodgedbars.pdf', dpi=300)dd